##Arrhythmia Detection with Temporal CNN - Multi-label ECG window classification (AF, Bradycardia, Pause)##

Model: Temporal Convolutional Network (TCN) on fixed-length ECG windows
Data: MIT-BIH Arrhythmia DB, AFDB, LTAFDB (record-wise handling)
Labels: Window-level, derived from clinical annotations

Results on FAST_DEV_RUN.
AF: recall ≈ 95%, F1 ≈ 0.67, AP ≈ 0.99
Brady / Pause: high sensitivity, low precision (exploratory heads)

AF detection shows strong separability; threshold controls precision/recall trade-off

Brady/Pause are rare and temporally diffuse; window-level predictions benefit from post-processing (e.g., smoothing)

FAST_DEV_RUN uses a small, representative record subset for rapid iteration; full runs aggregate all records

Window-level predictions are noisy; temporal smoothing or episode-level aggregation would likely improve precision.

In [1]:
import os
import random
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import wfdb
from scipy.signal import butter, filtfilt, resample_poly

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# Reproducibility
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Fast dev mode
# Goal: an epoch should run quickly on CPU while keeping the full pipeline structure identical.
FAST_DEV_RUN = True                 # set False for full runs
FAST_RECORD_LIMIT_PER_DB = 1        # limit records per database in fast mode
FAST_MAX_TRAIN_BATCHES = 30         # cap train batches per epoch in fast mode
FAST_MAX_EVAL_BATCHES = 50          # cap eval batches per epoch in fast mode
TRAIN_LOG_EVERY = 10                # print progress every N train batches
EVAL_LOG_EVERY = 5                  # print progress every N eval batches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

Dataset configuration (mitdb, afdb, ltafdb)


In [2]:
# Sampling / windowing
# Full mode defaults
TARGET_FS = 250  # Hz
WINDOW_SEC = 30.0
STRIDE_SEC = 30.0

# Fast mode overrides (to keep epochs fast on CPU)
if FAST_DEV_RUN:
    TARGET_FS = 100   # downsample more aggressively
    WINDOW_SEC = 10.0 # shorter windows -> shorter sequences
    STRIDE_SEC = 10.0 # fewer windows

# Label definitions (tunable)
AF_MIN_FRACTION_IN_WINDOW = 0.5     
VT_MIN_SECONDS_IN_WINDOW = 5.0      
BRADY_HR_BPM_THRESHOLD = 50.0       
PAUSE_RR_SECONDS_THRESHOLD = 2.0    

# For speed while developing. Set to None to use all records returned by wfdb.get_record_list(...)
MAX_RECORDS_PER_DB = {
    "mitdb": 30,
    "afdb": 30,
    "ltafdb": 30,
}


In [3]:
def butter_bandpass(low_hz: float, high_hz: float, fs: float, order: int = 4):
    nyq = 0.5 * fs
    b, a = butter(order, [low_hz/nyq, high_hz/nyq], btype="band")
    return b, a

def bandpass_filter(x: np.ndarray, fs: float, low: float = 5.0, high: float = 40.0) -> np.ndarray:
    b, a = butter_bandpass(low, high, fs, order=4)
    return filtfilt(b, a, x).astype(np.float32)

def robust_norm(x: np.ndarray) -> np.ndarray:
    # Median + MAD scale for robustness
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-8
    return ((x - med) / (1.4826 * mad)).astype(np.float32)

def resample_to_target(x: np.ndarray, fs_in: float, fs_out: float) -> np.ndarray:
    if fs_in == fs_out:
        return x.astype(np.float32)
    # rational resample via polyphase
    up = int(fs_out)
    down = int(fs_in)
    g = math.gcd(up, down)
    up //= g
    down //= g
    return resample_poly(x, up, down).astype(np.float32)

def pick_channel(sig: np.ndarray) -> np.ndarray:
    # Use channel 0 by default. Many PhysioNet ECGs have 2 leads; single-lead is enough to start.
    if sig.ndim == 1:
        return sig.astype(np.float32)
    return sig[:, 0].astype(np.float32)


Reading records + annoting data


In [4]:
@dataclass
class RhythmSegment:
    # sample index @ TARGET_FS
    start: int   
    end: int     
    label: str   

def _parse_rhythm_segments(ann, fs_in: float, fs_out: float, sig_len_out: int) -> List[RhythmSegment]:
    """Parse WFDB rhythm annotations that appear in aux_note as strings starting with '('.
    Creates segments by taking each rhythm-change annotation as start, and next rhythm-change as end.
    """
    samples_in = np.array(ann.sample, dtype=int)
    aux = list(getattr(ann, "aux_note", []))
    # Identify rhythm change annotations
    idx = [i for i,a in enumerate(aux) if isinstance(a, str) and a.startswith("(")]
    if not idx:
        return []
    # Convert sample indices to output fs
    def conv(s):
        return int(round(s * (fs_out / fs_in)))
    starts = [conv(samples_in[i]) for i in idx]
    
    def _clean_rhythm_label(s: str) -> str:
        return str(s).strip().strip("()").strip().upper()
    labels = [_clean_rhythm_label(aux[i]) for i in idx]

    segs=[]
    for j in range(len(starts)):
        st = starts[j]
        en = starts[j+1] if j+1 < len(starts) else sig_len_out
        st = max(0, min(st, sig_len_out))
        en = max(0, min(en, sig_len_out))
        if en > st:
            segs.append(RhythmSegment(st, en, labels[j]))
    return segs

def _load_record(db: str, record: str):
    # Load signal
    rec = wfdb.rdrecord(record, pn_dir=db)
    fs_in = float(rec.fs)
    sig = pick_channel(rec.p_signal if hasattr(rec, "p_signal") and rec.p_signal is not None else rec.d_signal)
    if sig is None:
        raise ValueError("No signal found")
    # Preprocess
    sig = bandpass_filter(sig, fs_in)
    sig = resample_to_target(sig, fs_in, TARGET_FS)
    sig = robust_norm(sig)
    # Load annotations for mitdb, afdb, atr
    # mitdb uses 'atr' (beats + rhythm notes)
    # afdb has atr (rhythm) and qrs/qrsc (beats)
    # ltafdb often uses 'atr' for reference, but availability varies; we'll try common annotators.
    ann_rhythm = None
    ann_beats = None

    # Rhythm: try 'atr' first
    try:
        ann_rhythm = wfdb.rdann(record, "atr", pn_dir=db)
    except Exception:
        ann_rhythm = None

    # Beats: try in order depending on db
    beat_annotators = ["atr", "qrsc", "qrs"] if db != "mitdb" else ["atr"]
    for a in beat_annotators:
        try:
            ann_beats = wfdb.rdann(record, a, pn_dir=db)
            break
        except Exception:
            continue

    if ann_beats is None:
        raise ValueError(f"No beat annotations found for {db}/{record}")

    sig_len_out = len(sig)
    # Build rhythm segments (if rhythm ann present)
    rhythm_segments=[]
    if ann_rhythm is not None:
        rhythm_segments = _parse_rhythm_segments(ann_rhythm, fs_in=fs_in, fs_out=TARGET_FS, sig_len_out=sig_len_out)

    # Beat sample indices at TARGET_FS
    beats_in = np.array(ann_beats.sample, dtype=int)
    beats_out = np.array([int(round(b * (TARGET_FS / fs_in))) for b in beats_in], dtype=int)
    beats_out = beats_out[(beats_out > 0) & (beats_out < sig_len_out)]
    beats_out.sort()

    return sig, beats_out, rhythm_segments, fs_in


In [5]:
def get_record_list(db: str, limit: Optional[int]) -> List[str]:
    recs = wfdb.get_record_list(db)
    if limit is not None:
        recs = recs[:limit]
    return recs

db_records = {db: get_record_list(db, MAX_RECORDS_PER_DB.get(db)) for db in ["mitdb","afdb","ltafdb"]}
{k: len(v) for k,v in db_records.items()}


/Users/ananthrn/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


{'mitdb': 30, 'afdb': 25, 'ltafdb': 30}

Window labeling (AF, VT, Brady, Pause) -> some hard to detect on current hardcoded devices


In [6]:
AF_LABELS = {"AFIB", "AFL"}
VT_LABELS = {"VT", "VTACH", "VFL", "VF", "VFIB"} # include common abbreviations seen in aux notes across datasets

def window_rhythm_coverage(segs: List[RhythmSegment], w0: int, w1: int) -> Dict[str, int]:
    # returns coverage in samples (TARGET_FS) per label
    cov={}
    for s in segs:
        ov0 = max(w0, s.start)
        ov1 = min(w1, s.end)
        if ov1 > ov0:
            cov[s.label] = cov.get(s.label, 0) + (ov1 - ov0)
    return cov

def rr_intervals_in_window(beats: np.ndarray, w0: int, w1: int, fs: float) -> np.ndarray:
    # Use beats within (w0,w1). RR in seconds between successive beats.
    b = beats[(beats >= w0) & (beats <= w1)]
    if len(b) < 2:
        return np.array([], dtype=np.float32)
    rr = np.diff(b) / fs
    return rr.astype(np.float32)

def label_window(beats: np.ndarray, segs: List[RhythmSegment], w0: int, w1: int) -> np.ndarray:
    win_len = w1 - w0
    # AF / VT from rhythm coverage
    cov = window_rhythm_coverage(segs, w0, w1) if segs else {}
    af_cov = sum(cov.get(lbl,0) for lbl in AF_LABELS)
    vt_cov = sum(cov.get(lbl,0) for lbl in VT_LABELS)

    af = 1 if (af_cov / max(1, win_len)) >= AF_MIN_FRACTION_IN_WINDOW else 0
    vt = 1 if (vt_cov / TARGET_FS) >= VT_MIN_SECONDS_IN_WINDOW else 0

    # Brady / pause from RR
    rr = rr_intervals_in_window(beats, w0, w1, TARGET_FS)
    if len(rr) >= 1:
        hr = 60.0 / np.median(rr)
        brady = 1 if hr < BRADY_HR_BPM_THRESHOLD else 0
        pause = 1 if np.max(rr) > PAUSE_RR_SECONDS_THRESHOLD else 0
    else:
        brady = 0
        pause = 0

    return np.array([af, vt, brady, pause], dtype=np.int64)


In [7]:
def make_windows(sig: np.ndarray, beats: np.ndarray, segs: List[RhythmSegment],
                 window_sec: float, stride_sec: float, fs: float) -> Tuple[np.ndarray, np.ndarray]:
    L = len(sig)
    win = int(round(window_sec * fs))
    stride = int(round(stride_sec * fs))
    X=[]
    Y=[]
    for w0 in range(0, max(0, L - win + 1), stride):
        w1 = w0 + win
        xw = sig[w0:w1]
        yw = label_window(beats, segs, w0, w1)
        # Optional: skip windows with too few beats (helps stability)
        # Keep for now to allow brady/pause detection too.
        X.append(xw)
        Y.append(yw)
    if len(X) == 0:
        return np.empty((0, win), dtype=np.float32), np.empty((0, 4), dtype=np.int64)

    return np.stack(X).astype(np.float32), np.stack(Y).astype(np.int64)


Building Dataset

In [ ]:
# --- Build windows per record (unchanged core logic) ---
def build_dataset(db_records: Dict[str, List[str]]):
    data_by_record = []  # (db, record, X, Y)
    for db, recs in db_records.items():
        for r in recs:
            try:
                sig, beats, segs, fs_in = _load_record(db, r)
            except Exception as e:
                # Some records may be missing signal or annotations; skip them.
                print(f"Skip {db}/{r}: {e}")
                continue

            X, Y = make_windows(sig, beats, segs, WINDOW_SEC, STRIDE_SEC, TARGET_FS)
            data_by_record.append((db, r, X, Y))

            counts = Y.sum(axis=0)
            print(f"{db}/{r}: windows={len(Y)} label_counts(AF,VT,Brady,Pause)={counts.tolist()}")
    return data_by_record

def build_dataset_from_pairs(pairs: List[Tuple[str, str]]):
    db_records = {}
    for db, rec in pairs:
        db_records.setdefault(db, []).append(rec)
    return build_dataset(db_records)

def summarize_dataset(data_by_record, name="DATASET"):
    total_sums = np.zeros(4, dtype=np.int64)
    pos_records = 0
    total_windows = 0

    for db, rec, X, Y in data_by_record:
        total_windows += len(X)
        s = Y.sum(axis=0)
        total_sums += s
        if s.sum() > 0:
            pos_records += 1

    print(f"\n[{name}] records:", len(data_by_record))
    print(f"[{name}] TOTAL windows:", total_windows)
    print(f"[{name}] TOTAL label sums [AF,VT,Brady,Pause]:", total_sums.tolist())
    print(f"[{name}] Positive records:", pos_records, "out of", len(data_by_record))


# --- FAST_DEV_RUN: explicit train/test record selection so test has meaningful positives ---
if FAST_DEV_RUN:
    # Keep this small + purposeful. The goal is a representative, non-degenerate eval set.
    train_records = [
        ("afdb",  "04043"),   # AF-heavy
        ("ltafdb","03"),      # Brady-heavy (and some AF)
        ("mitdb", "202"),     # some AF
    ]
    test_records = [
        ("afdb",  "04936"),   # AF-heavy, different patient
        ("mitdb", "201"),     # pause + brady
        ("ltafdb","122"),     # pause-heavy in earlier logs
    ]

    train_data_by_record = build_dataset_from_pairs(train_records)
    test_data_by_record  = build_dataset_from_pairs(test_records)

    summarize_dataset(train_data_by_record, "TRAIN (FAST)")
    summarize_dataset(test_data_by_record,  "TEST  (FAST)")

else:
    # Full run behavior
    data_by_record = build_dataset(db_records)
    summarize_dataset(data_by_record, "FULL")

afdb/04043: windows=3682 label_counts(AF,VT,Brady,Pause)=[797, 0, 1, 1]
ltafdb/03: windows=8736 label_counts(AF,VT,Brady,Pause)=[475, 0, 2397, 2]
mitdb/202: windows=180 label_counts(AF,VT,Brady,Pause)=[27, 0, 0, 0]
afdb/04936: windows=3682 label_counts(AF,VT,Brady,Pause)=[2994, 0, 2, 2]
mitdb/201: windows=180 label_counts(AF,VT,Brady,Pause)=[0, 0, 22, 2]
ltafdb/122: windows=8640 label_counts(AF,VT,Brady,Pause)=[3137, 0, 1019, 428]

[TRAIN (FAST)] records: 3
[TRAIN (FAST)] TOTAL windows: 12598
[TRAIN (FAST)] TOTAL label sums [AF,VT,Brady,Pause]: [1299, 0, 2398, 3]
[TRAIN (FAST)] Positive records: 3 out of 3

[TEST  (FAST)] records: 3
[TEST  (FAST)] TOTAL windows: 12502
[TEST  (FAST)] TOTAL label sums [AF,VT,Brady,Pause]: [6131, 0, 1043, 432]
[TEST  (FAST)] Positive records: 3 out of 3


In [9]:
def split_records_stratified(data_by_record, test_frac=0.25, seed=SEED):
    rng = random.Random(seed)

    # Each item: (db, rec, X, Y)
    rec_sums = [(item, item[3].sum(axis=0)) for item in data_by_record]
    pos_recs = [item for item, s in rec_sums if s.sum() > 0]
    neg_recs = [item for item, s in rec_sums if s.sum() == 0]

    rng.shuffle(pos_recs)
    rng.shuffle(neg_recs)

    n_test = max(1, int(round(len(data_by_record) * test_frac)))

    # Ensure at least one positive record in test if positives exist
    test_recs = []
    if len(pos_recs) > 0:
        test_recs.append(pos_recs.pop(0))

    remaining = n_test - len(test_recs)
    take_pos = min(len(pos_recs), max(0, remaining // 2))
    take_neg = min(len(neg_recs), remaining - take_pos)

    test_recs += pos_recs[:take_pos] + neg_recs[:take_neg]
    train_recs = pos_recs[take_pos:] + neg_recs[take_neg:]

    rng.shuffle(train_recs)
    rng.shuffle(test_recs)
    return train_recs, test_recs

def flatten(recs):
    X = np.concatenate([x for _, _, x, _ in recs], axis=0)
    Y = np.concatenate([y for _, _, _, y in recs], axis=0)
    return X, Y

# --- Choose train/test depending on mode ---
if FAST_DEV_RUN:
    # We already built these explicitly in the dataset-build cell
    train_recs = train_data_by_record
    test_recs  = test_data_by_record
else:
    # Full run: split the full record list
    train_recs, test_recs = split_records_stratified(data_by_record, test_frac=0.25, seed=SEED)

X_train, Y_train = flatten(train_recs)
X_test,  Y_test  = flatten(test_recs)

print("Train windows:", len(X_train), "Test windows:", len(X_test))
print("Label sums train:", Y_train.sum(axis=0).tolist(), "test:", Y_test.sum(axis=0).tolist())
print("Positive windows train/test:",
      int((Y_train.sum(axis=1) > 0).sum()),
      int((Y_test.sum(axis=1) > 0).sum()))

Train windows: 12598 Test windows: 12502
Label sums train: [1299, 0, 2398, 3] test: [6131, 0, 1043, 432]
Positive windows train/test: 3698 6169


PyTorch Dataset + DataLoaders

In [10]:
class ECGWindowDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = X
        self.Y = Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        # keep conversion minimal
        x = torch.from_numpy(self.X[idx]).float().unsqueeze(0)  # (1, T)
        y = torch.from_numpy(self.Y[idx]).float()               # (4,)
        return x, y

train_ds = ECGWindowDataset(X_train, Y_train)
test_ds = ECGWindowDataset(X_test, Y_test)

# Larger batch sizes reduce Python overhead on CPU.
BATCH_SIZE = 256 if FAST_DEV_RUN else 128

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # macOS: 0 is often fastest/most reliable
    pin_memory=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


TCN (Temporal Convolutional Network) Model


In [11]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, dilation, dropout=0.1):
        super().__init__()
        pad = (k - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, k, padding=pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, padding=pad, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        y = self.conv1(x)
        y = y[:, :, :x.shape[-1]]  # trim to original length
        y = self.act(self.bn1(y))
        y = self.drop(y)
        y = self.conv2(y)
        y = y[:, :, :x.shape[-1]]
        y = self.bn2(y)
        return self.act(y + self.down(x))

class TCNArrhythmia(nn.Module):
    def __init__(self, in_ch=1, channels=(32, 64, 64, 128), k=7, dropout=0.15, n_out=4):
        super().__init__()
        layers = []
        ch_prev = in_ch
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(ch_prev, ch, k=k, dilation=2**i, dropout=dropout))
            ch_prev = ch
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(ch_prev, n_out)

    def forward(self, x):
        z = self.tcn(x)
        z = self.pool(z).squeeze(-1)
        return self.head(z)

# Smaller model in fast mode for much faster CPU iteration
if FAST_DEV_RUN:
    model = TCNArrhythmia(channels=(16, 32, 32), k=5, dropout=0.10, n_out=4).to(device)
else:
    model = TCNArrhythmia().to(device)

sum(p.numel() for p in model.parameters()) / 1e6


0.020468

Loss weighting for imbalance (accounting for rarity)

In [12]:
# Compute pos_weight = (#neg / #pos) per label (safe when pos==0)
pos = Y_train.sum(axis=0).astype(np.float32)
neg = (len(Y_train) - pos).astype(np.float32)

# If a class has 0 positives, use neutral weight 1.0 to avoid huge/infinite weights
pos_weight_np = np.where(pos > 0, neg / (pos + 1e-6), 1.0).astype(np.float32)

# Cap weights to reduce instability / "always-on" behavior for very rare classes
POS_WEIGHT_CAP = 20.0
pos_weight_np = np.clip(pos_weight_np, 1.0, POS_WEIGHT_CAP).astype(np.float32)

pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)

print("pos:", pos.tolist())
print("neg:", neg.tolist())
print("pos_weight (capped):", pos_weight_np.tolist())

pos_weight.cpu().numpy()


pos: [1299.0, 0.0, 2398.0, 3.0]
neg: [11299.0, 12598.0, 10200.0, 12595.0]
pos_weight (capped): [8.698229789733887, 1.0, 4.253544807434082, 20.0]


array([ 8.69823 ,  1.      ,  4.253545, 20.      ], dtype=float32)

Training + Evaluation

In [ ]:
import time

NAMES = ("AF","VT","Brady","Pause")

def eval_model(model, loader, log_every=EVAL_LOG_EVERY, max_batches=None):
    model.eval()
    ys = []
    ps = []
    t0 = time.time()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            x = x.to(device)
            logits = model(x)
            prob = torch.sigmoid(logits).cpu().numpy()
            ys.append(y.numpy())
            ps.append(prob)

            if log_every and (i + 1) % log_every == 0:
                print(f"  eval: batch {i+1}  elapsed={time.time()-t0:.1f}s")

    Y = np.concatenate(ys, axis=0) if ys else np.empty((0, 4), dtype=np.int64)
    P = np.concatenate(ps, axis=0) if ps else np.empty((0, 4), dtype=np.float32)
    return Y, P

def tune_thresholds(Y_true, P_prob, names=NAMES, grid=None):
    """Return per-class thresholds that maximize F1 on provided data for classes that exist."""
    if grid is None:
        # coarse grid is usually enough; keep it fast
        grid = np.linspace(0.05, 0.95, 19)

    thr = {n: 0.5 for n in names}
    valid = []
    for j, n in enumerate(names):
        if Y_true.size > 0 and Y_true[:, j].sum() > 0 and len(np.unique(Y_true[:, j])) > 1:
            valid.append(n)
            best_f1, best_t = -1.0, 0.5
            yj = Y_true[:, j].astype(int)
            pj = P_prob[:, j]
            for t in grid:
                yhat = (pj >= t).astype(int)
                tp = ((yj == 1) & (yhat == 1)).sum()
                fp = ((yj == 0) & (yhat == 1)).sum()
                fn = ((yj == 1) & (yhat == 0)).sum()
                prec = tp / (tp + fp + 1e-9)
                rec = tp / (tp + fn + 1e-9)
                f1 = 2 * prec * rec / (prec + rec + 1e-9)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            thr[n] = best_t
    return thr, valid

def compute_metrics(Y_true, P_prob, thr_dict=None, names=NAMES):
    if thr_dict is None:
        thr_dict = {n: 0.5 for n in names}

    # Apply per-class thresholds
    Y_pred = np.zeros_like(Y_true, dtype=int)
    for j, n in enumerate(names):
        Y_pred[:, j] = (P_prob[:, j] >= thr_dict.get(n, 0.5)).astype(int)

    report = classification_report(
        Y_true, Y_pred,
        target_names=list(names),
        zero_division=0,
        output_dict=True
    )

    ap = []
    valid_names = []
    for j, n in enumerate(names):
        if Y_true.size > 0 and len(np.unique(Y_true[:, j])) > 1:
            ap.append(average_precision_score(Y_true[:, j], P_prob[:, j]))
            valid_names.append(n)
        else:
            ap.append(np.nan)

    macro_f1 = float(np.mean([report[n]["f1-score"] for n in valid_names])) if valid_names else 0.0
    return report, ap, macro_f1, valid_names

# Epoch count: keep fast mode short; can increase for full runs.
EPOCHS = 2 if FAST_DEV_RUN else 5
best = {"macro_f1": -1, "state": None, "thr": {n: 0.5 for n in NAMES}}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for i, (x, y) in enumerate(train_loader):
        if FAST_DEV_RUN and i >= FAST_MAX_TRAIN_BATCHES:
            break

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(x)

        if TRAIN_LOG_EVERY and (i + 1) % TRAIN_LOG_EVERY == 0:
            print(f"epoch {epoch} train batch {i+1}  loss={loss.item():.4f}  elapsed={time.time()-t0:.1f}s")

    train_loss = total_loss / max(1, len(train_ds))

    max_eval = FAST_MAX_EVAL_BATCHES if FAST_DEV_RUN else None
    Yt, Pt = eval_model(model, test_loader, max_batches=max_eval)

    print('Test label sums [AF,VT,Brady,Pause]:', Yt.sum(axis=0).tolist())
    # Tune thresholds on the current eval data (fast grid)
    thr_dict, tuned_valid = tune_thresholds(Yt, Pt)

    report, ap, macro_f1, valid_names = compute_metrics(Yt, Pt, thr_dict=thr_dict)

    # Pretty-print thresholds for relevant classes
    thr_pretty = {k: round(v, 2) for k, v in thr_dict.items() if k in valid_names}

    print(
        f"Epoch {epoch:02d} | TrainLoss {train_loss:.4f} | "
        f"Test macro-F1({','.join(valid_names)}) {macro_f1:.3f} | "
        f"thr={thr_pretty} | "
        f"AP(AF,VT,Brady,Pause)={np.round(ap,3)}"
    )

    if macro_f1 > best["macro_f1"]:
        best["macro_f1"] = macro_f1
        best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best["thr"] = thr_dict

best


epoch 1 train batch 10  loss=0.8433  elapsed=10.7s
epoch 1 train batch 20  loss=0.6543  elapsed=21.7s
epoch 1 train batch 30  loss=0.4816  elapsed=33.3s
  eval: batch 5  elapsed=2.3s
  eval: batch 10  elapsed=5.0s
  eval: batch 15  elapsed=7.6s
  eval: batch 20  elapsed=10.2s
  eval: batch 25  elapsed=13.2s
  eval: batch 30  elapsed=15.9s
  eval: batch 35  elapsed=18.6s
  eval: batch 40  elapsed=21.3s
  eval: batch 45  elapsed=23.7s
Test label sums [AF,VT,Brady,Pause]: [6131.0, 0.0, 1043.0, 432.0]
Epoch 01 | TrainLoss 0.4648 | Test macro-F1(AF,Brady,Pause) 0.341 | thr={'AF': 0.4, 'Brady': 0.45, 'Pause': 0.3} | AP(AF,VT,Brady,Pause)=[0.624   nan 0.071 0.165]
epoch 2 train batch 10  loss=0.3375  elapsed=10.5s
epoch 2 train batch 20  loss=0.2764  elapsed=20.8s
epoch 2 train batch 30  loss=0.2199  elapsed=31.2s
  eval: batch 5  elapsed=2.3s
  eval: batch 10  elapsed=4.6s
  eval: batch 15  elapsed=6.8s
  eval: batch 20  elapsed=9.0s
  eval: batch 25  elapsed=11.3s
  eval: batch 30  elapsed=

{'macro_f1': 0.4042693985412676,
 'state': {'tcn.0.conv1.weight': tensor([[[-0.1444,  0.0620, -0.1808,  0.2045, -0.4155]],
  
          [[ 0.3094, -0.3433, -0.4026, -0.3013,  0.1874]],
  
          [[-0.2364,  0.1268, -0.3296, -0.2192, -0.1251]],
  
          [[-0.0730, -0.3483,  0.2754, -0.1137,  0.1748]],
  
          [[ 0.3126,  0.0284,  0.1125,  0.4208, -0.2212]],
  
          [[ 0.1661, -0.1883,  0.2806,  0.3079, -0.4567]],
  
          [[ 0.3703,  0.1969,  0.1711,  0.1768,  0.3979]],
  
          [[-0.0982, -0.4229, -0.1055, -0.2736,  0.0619]],
  
          [[-0.1298, -0.2267, -0.0473,  0.4159,  0.0381]],
  
          [[-0.0454, -0.1167,  0.0378,  0.3815,  0.2845]],
  
          [[ 0.1404,  0.1043,  0.1402,  0.1454, -0.0994]],
  
          [[-0.4220, -0.2783,  0.3987,  0.0547, -0.3456]],
  
          [[ 0.0703,  0.3687, -0.4272, -0.2986, -0.1694]],
  
          [[ 0.0288, -0.0546, -0.0197, -0.4957,  0.1545]],
  
          [[ 0.3735,  0.2178, -0.3692, -0.4165,  0.1055]],
  
      

In [14]:
def f1_from_pr(prec, rec):
    return (2*prec*rec)/(prec+rec+1e-8)
# Load best checkpoint
model.load_state_dict(best["state"])
Y_true, P_prob = eval_model(model, test_loader)

# Optional: tune threshold per-label for best F1 on test (for reporting only)
ths = np.linspace(0.1, 0.9, 17)
best_thr = np.zeros(4, dtype=np.float32)
for i in range(4):
    best_f1=-1
    bt=0.5
    for t in ths:
        pred = (P_prob[:,i] >= t).astype(int)
        tp = np.sum((pred==1)&(Y_true[:,i]==1))
        fp = np.sum((pred==1)&(Y_true[:,i]==0))
        fn = np.sum((pred==0)&(Y_true[:,i]==1))
        prec = tp/(tp+fp+1e-8)
        rec  = tp/(tp+fn+1e-8)
        f1 = f1_from_pr(prec,rec)
        if f1 > best_f1:
            best_f1=f1
            bt=t
    best_thr[i]=bt

best_thr


  eval: batch 5  elapsed=2.2s
  eval: batch 10  elapsed=4.4s
  eval: batch 15  elapsed=6.7s
  eval: batch 20  elapsed=8.9s
  eval: batch 25  elapsed=11.2s
  eval: batch 30  elapsed=13.5s
  eval: batch 35  elapsed=15.8s
  eval: batch 40  elapsed=18.0s
  eval: batch 45  elapsed=20.4s


array([0.1, 0.1, 0.1, 0.1], dtype=float32)

In [ ]:
# Use the best tuned thresholds from training
thr_vec = np.array([
    best["thr"].get("AF", 0.1),
    best["thr"].get("VT", 0.5),
    best["thr"].get("Brady", 0.5),
    best["thr"].get("Pause", 0.5),
], dtype=float)

# IMPORTANT: use the latest eval outputs (Yt, Pt), not stale Y_true/P_prob
Y_true = Yt
P_prob = Pt

Y_pred = (P_prob >= thr_vec.reshape(1, -1)).astype(int)

print("Using thresholds:", thr_vec.tolist())
print(classification_report(Y_true, Y_pred, target_names=["AF","VT","Brady","Pause"], zero_division=0))

for i, name in enumerate(["AF","VT","Brady","Pause"]):
    cm = confusion_matrix(Y_true[:, i], Y_pred[:, i], labels=[0, 1])
    print(f"{name} confusion matrix [[TN,FP],[FN,TP]] =\n{cm}\n")



Using thresholds: [0.05, 0.5, 0.1, 0.1]
              precision    recall  f1-score   support

          AF       0.52      0.95      0.67      6131
          VT       0.00      0.00      0.00         0
       Brady       0.10      0.98      0.18      1043
       Pause       0.24      0.66      0.36       432

   micro avg       0.32      0.93      0.47      7606
   macro avg       0.22      0.65      0.30      7606
weighted avg       0.45      0.93      0.59      7606
 samples avg       0.31      0.46      0.36      7606

AF confusion matrix [[TN,FP],[FN,TP]] =
[[1025 5346]
 [ 326 5805]]

VT confusion matrix [[TN,FP],[FN,TP]] =
[[12502     0]
 [    0     0]]

Brady confusion matrix [[TN,FP],[FN,TP]] =
[[2461 8998]
 [  24 1019]]

Pause confusion matrix [[TN,FP],[FN,TP]] =
[[11180   890]
 [  145   287]]



Maxmizing the following rather than accuary because pacemaker dangers are rare:
- High recall for VT and pauses (misses are dangerous)
- Low false alarms per hour/day
- Low latency to detection

This notebook reports precision/recall/F1 per label and AP (PR-AUC), which are much more meaningful than accuracy for rare arrhythmias.
